# Chapter 1 — Foundations (R notebook)

> Companion to [`notes.md`](../notes.md), [`worked-examples.md`](../worked-examples.md), and [`exercises.md`](../exercises.md).

This notebook has six sections, each tied to a specific section of `notes.md`:

1. **Anchor problem (§1.0, §1.12):** the COVID base-rate simulation.
2. **Counting (§1.4–§1.5):** the birthday problem, poker hand probabilities.
3. **Axioms in action (§1.7–§1.8):** inclusion–exclusion via Monte Carlo.
4. **Conditional probability, Bayes, and independence (§1.9–§1.13):** a 2×2 table, two-coin-flip independence.
5. **Pitfalls (§1.14):** Monty Hall simulation, Simpson's paradox table.
6. **Short R exercises with solutions.**

## Setup

This notebook uses the **R kernel** (IRkernel). If you see a Python kernel in the top right, pick *Kernel → Change Kernel… → R*. Install IRkernel once from an R console with:

```R
install.packages(c("IRkernel", "tidyverse", "ggplot2", "gridExtra"))
IRkernel::installspec()
```


In [ ]:
# Libraries. tidyverse brings ggplot2 + dplyr + tibble; gridExtra helps side-by-side plots.
suppressPackageStartupMessages({
  library(tidyverse)
  library(gridExtra)
})

# Reproducible random streams for every simulation in this notebook.
set.seed(12345)

## 1. The anchor problem — COVID false positives (§1.0, §1.12)

A test with 99% sensitivity and 99% specificity is applied in a population with prevalence `P(D+)`. We compute the posterior `P(D− | T+)` and show that the “60% of positives are false” headline is compatible with the test — in a low-prevalence population.

In [ ]:
# Bayes' rule for the COVID test.
bayes_dneg_given_tpos <- function(prior, sens = 0.99, spec = 0.99) {
  p_tpos <- sens * prior + (1 - spec) * (1 - prior)
  (1 - spec) * (1 - prior) / p_tpos
}

# Table: posterior P(D- | T+) as prevalence sweeps from 0.1% to 20%.
prev_grid <- c(0.001, 0.005, 0.01, 0.02, 0.05, 0.10, 0.20)
posterior <- sapply(prev_grid, bayes_dneg_given_tpos)
tibble(prevalence = prev_grid,
       `P(D- | T+)` = round(posterior, 4))

In [ ]:
# Plot posterior vs. prevalence. Dashed line at 0.6 marks the headline.
prev <- seq(0.001, 0.2, length.out = 400)
df <- tibble(prevalence = prev,
             posterior  = bayes_dneg_given_tpos(prev))

ggplot(df, aes(prevalence, posterior)) +
  geom_line(linewidth = 1) +
  geom_hline(yintercept = 0.6, linetype = "dashed", colour = "firebrick") +
  annotate("text", x = 0.15, y = 0.63, label = "headline: 60% false",
           colour = "firebrick", size = 3.5) +
  scale_x_continuous(labels = scales::percent_format(accuracy = 0.1)) +
  scale_y_continuous(labels = scales::percent_format(accuracy = 1),
                     limits = c(0, 1)) +
  labs(title    = "P(D- | T+) as a function of population prevalence",
       subtitle = "Test sensitivity = 99%, specificity = 99%",
       x = "Prevalence P(D+)",
       y = "Posterior P(D- | T+)   — fraction of positives that are false")

In [ ]:
# Monte Carlo: 100,000 patients at 1% prevalence.
# Reproduce the analytic 0.5 answer the hard way, as a sanity check.
N <- 1e5
prior <- 0.01
sens  <- 0.99
spec  <- 0.99

disease <- rbinom(N, 1, prior)                         # 1 = D+, 0 = D-
test    <- ifelse(disease == 1,
                  rbinom(N, 1, sens),                  # P(T+|D+) = sens
                  rbinom(N, 1, 1 - spec))              # P(T+|D-) = 1 - spec

positives           <- which(test == 1)
false_positive_rate <- mean(disease[positives] == 0)

cat(sprintf("At prevalence %.1f%%: %d positives; fraction that are false = %.4f\n",
            prior * 100, length(positives), false_positive_rate))
cat(sprintf("Analytic Bayes answer: %.4f\n",
            bayes_dneg_given_tpos(prior, sens, spec)))

## 2. Counting — birthday problem and poker hands (§1.4–§1.5)

Two set pieces of elementary probability:

- **Birthday problem**: in a room of `k` people, the probability that at least two share a birthday exceeds 50% at `k = 23`. The complement rule is the key.
- **Poker**: probabilities of four-of-a-kind, full house, flush, etc. — all derived from `choose(52, 5) = 2,598,960` five-card hands.

In [ ]:
# Birthday problem: P(no match) = 365*364*...*(366-k) / 365^k,
# so P(at least one match) = 1 - prod.
p_shared_birthday <- function(k) 1 - prod((365 - 0:(k-1)) / 365)

cat(sprintf("P(shared) at k=23: %.6f\n", p_shared_birthday(23)))
cat(sprintf("P(shared) at k=50: %.6f\n", p_shared_birthday(50)))

df_bday <- tibble(k = 2:60,
                  p = sapply(2:60, p_shared_birthday))

ggplot(df_bday, aes(k, p)) +
  geom_line(linewidth = 1) +
  geom_hline(yintercept = 0.5, linetype = "dashed") +
  geom_vline(xintercept = 23, linetype = "dashed", colour = "steelblue") +
  scale_y_continuous(labels = scales::percent_format(accuracy = 1)) +
  labs(title = "Birthday problem: P(at least two share a birthday)",
       subtitle = "The 50% threshold is crossed at k = 23",
       x = "Number of people k",
       y = "P(shared birthday)")

In [ ]:
# Poker hand probabilities.
n_hands <- choose(52, 5)   # 2,598,960

# Four-of-a-kind: choose rank (13) x choose kicker-rank (12) x choose kicker-suit (4)
p_4kind <- 13 * 12 * 4 / n_hands

# Full house: choose rank for trip (13) x pick 3 of 4 suits (choose(4,3))
#             x choose different rank for pair (12) x pick 2 of 4 suits (choose(4,2))
p_fh <- 13 * choose(4, 3) * 12 * choose(4, 2) / n_hands

# Flush (including straight flush): 4 suits x choose 5 cards from 13
p_flush <- 4 * choose(13, 5) / n_hands

# One pair exactly: choose rank (13) x pick 2 of 4 suits
#   x choose 3 other distinct ranks (choose(12,3)) x pick 1 suit each (4^3)
p_one_pair <- 13 * choose(4, 2) * choose(12, 3) * 4^3 / n_hands

tibble(hand = c("Four of a kind", "Full house", "Flush (any)", "One pair exactly"),
       probability = c(p_4kind, p_fh, p_flush, p_one_pair),
       `1-in` = round(1 / c(p_4kind, p_fh, p_flush, p_one_pair))) |>
  mutate(probability = sprintf("%.6f", probability))

## 3. Axioms in action — inclusion–exclusion by simulation (§1.7–§1.8)

The two-set inclusion–exclusion identity `P(A ∪ B) = P(A) + P(B) − P(A ∩ B)` lets us cross-check the birthday probability by direct counting. We also use Monte Carlo to sanity-check a three-event case.

In [ ]:
# 30 people. Let A = 'some match in the first 15', B = 'some match in the last 15'.
# Compute P(A ∪ B) analytically via inclusion-exclusion, then by simulation.

p_A <- p_shared_birthday(15)
p_B <- p_shared_birthday(15)

# P(A ∩ B): both halves have a collision, independently.
# Groups are disjoint (different people), so births are independent -> events independent.
p_AB <- p_A * p_B

p_union_analytic <- p_A + p_B - p_AB
cat(sprintf("Analytic  P(A ∪ B) = %.6f\n", p_union_analytic))

# Simulation.
N <- 1e5
sim_has_match <- function(k) {
  b <- sample.int(365, k, replace = TRUE)
  length(unique(b)) < k
}
sims <- replicate(N, {
  half1 <- sim_has_match(15)
  half2 <- sim_has_match(15)
  c(half1, half2, half1 || half2)
})
cat(sprintf("Simulated P(A)     = %.4f   (analytic %.4f)\n", mean(sims[1, ]), p_A))
cat(sprintf("Simulated P(B)     = %.4f   (analytic %.4f)\n", mean(sims[2, ]), p_B))
cat(sprintf("Simulated P(A ∪ B) = %.4f   (analytic %.4f)\n", mean(sims[3, ]), p_union_analytic))

## 4. Conditional probability, Bayes, independence (§1.9–§1.13)

### 2×2 table → conditional probabilities

A 2×2 table of (disease status × test result) makes the conditional arithmetic obvious. We generate a single hypothetical cohort of 1000 patients.

In [ ]:
N     <- 1000
prior <- 0.01
sens  <- 0.99
spec  <- 0.99

dplus  <- round(N * prior)
dminus <- N - dplus
tab <- matrix(
  c(round(dplus  * sens),        round(dplus  * (1 - sens)),    # D+ row: T+, T-
    round(dminus * (1 - spec)),  round(dminus * spec)),         # D- row: T+, T-
  nrow = 2, byrow = TRUE,
  dimnames = list(c("D+ (has COVID)", "D- (no COVID)"),
                  c("T+ (tests pos)", "T- (tests neg)"))
)
tab

In [ ]:
# Every conditional probability we might want, straight from the table.
total        <- sum(tab)
p_Dpos       <- sum(tab["D+ (has COVID)", ]) / total
p_Tpos       <- sum(tab[, "T+ (tests pos)"]) / total
p_Tpos_Dpos  <- tab["D+ (has COVID)", "T+ (tests pos)"] / sum(tab["D+ (has COVID)", ])
p_Dpos_Tpos  <- tab["D+ (has COVID)", "T+ (tests pos)"] / sum(tab[, "T+ (tests pos)"])
p_Dneg_Tpos  <- tab["D- (no COVID)",  "T+ (tests pos)"] / sum(tab[, "T+ (tests pos)"])

cat(sprintf("P(D+)        = %.4f  (set)\n",                  p_Dpos))
cat(sprintf("P(T+)        = %.4f\n",                         p_Tpos))
cat(sprintf("P(T+ | D+)   = %.4f  (sensitivity)\n",           p_Tpos_Dpos))
cat(sprintf("P(D+ | T+)   = %.4f  (posterior)\n",             p_Dpos_Tpos))
cat(sprintf("P(D- | T+)   = %.4f  (false-pos rate among positives)\n", p_Dneg_Tpos))

### Independence of two coin flips

`P(HH) = P(H) · P(H) = 1/4`. We confirm this two ways: a 2×2 count table, and a direct empirical comparison.

In [ ]:
# 10,000 pairs of independent coin flips.
N <- 1e4
flips <- tibble(
  flip1 = sample(c("H", "T"), N, replace = TRUE),
  flip2 = sample(c("H", "T"), N, replace = TRUE)
)

# Joint counts.
with(flips, table(flip1, flip2))

# Empirical P(HH) vs. P(H) * P(H).
p_H1 <- mean(flips$flip1 == "H")
p_H2 <- mean(flips$flip2 == "H")
p_HH <- mean(flips$flip1 == "H" & flips$flip2 == "H")
cat(sprintf("P(H₁) * P(H₂) = %.4f\nP(HH)         = %.4f  (should be ≈ equal)\n",
            p_H1 * p_H2, p_HH))

## 5. Classic pitfalls (§1.14)

### Monty Hall

Simulate 100,000 games of Monty Hall with two strategies: *stay* and *switch*. The analytic answer is 1/3 vs. 2/3.

In [ ]:
play_monty <- function(strategy = c("stay", "switch")) {
  strategy <- match.arg(strategy)
  car  <- sample.int(3, 1)        # where the car is
  pick <- sample.int(3, 1)        # player's initial pick (random)
  # Host opens a goat-door that isn't the player's pick.
  available_to_open <- setdiff(1:3, c(car, pick))
  open_door <- if (length(available_to_open) == 1) available_to_open
               else sample(available_to_open, 1)
  final <- if (strategy == "stay") pick
           else setdiff(1:3, c(pick, open_door))
  final == car
}

N <- 1e5
wins_stay   <- mean(replicate(N, play_monty("stay")))
wins_switch <- mean(replicate(N, play_monty("switch")))

cat(sprintf("Stay   : win rate = %.4f  (analytic 1/3 = %.4f)\n", wins_stay,   1/3))
cat(sprintf("Switch : win rate = %.4f  (analytic 2/3 = %.4f)\n", wins_switch, 2/3))

### Simpson's paradox

A small but real 2×2×2 example: two treatments, applied to mild vs. severe cases. Within each severity, treatment A is better — but pooled, B looks better, because B is disproportionately applied to mild cases.

In [ ]:
simpson <- tribble(
  ~severity, ~treatment, ~treated, ~recovered,
  "Mild",    "A",         87,       81,
  "Mild",    "B",         270,      234,
  "Severe",  "A",         263,      192,
  "Severe",  "B",         80,       55
)

# Within-severity recovery rates.
simpson |>
  mutate(rate = recovered / treated) |>
  select(severity, treatment, treated, recovered, rate)

In [ ]:
# Pooled rates.
simpson |>
  group_by(treatment) |>
  summarise(treated   = sum(treated),
            recovered = sum(recovered),
            rate      = recovered / treated,
            .groups   = "drop")

Within each severity, treatment **A** has the higher recovery rate (0.931 vs. 0.867 for mild; 0.730 vs. 0.688 for severe). But pooled, treatment **B** appears better (0.826 vs. 0.780) — because B was disproportionately applied to the easier-to-treat mild cases. The pooled number confounds *effectiveness of the treatment* with *which kinds of patients got the treatment*.

This is **Simpson's paradox** and the core reason we do randomized controlled trials: randomization balances the severity distribution between treatment arms so the pooled comparison is fair.

## 6. Short R exercises

Each has a solution immediately below. Try before peeking.

**R-1.** Estimate `P(at least one shared birthday)` in a room of 40 people by Monte Carlo with 10,000 replications. Compare to the analytic value.

**R-2.** A bag has 3 red and 2 blue balls. You draw 2 without replacement. Compute `P(both red)` analytically (using `choose`) and by simulation.

**R-3.** Suppose the COVID test has sensitivity 0.95 and specificity 0.90. For which prevalence `p` does `P(D− | T+) = 0.5`? Solve numerically with `uniroot`.

In [ ]:
# R-1 solution
set.seed(1)
sim_once <- function() {
  b <- sample.int(365, 40, replace = TRUE)
  length(unique(b)) < 40
}
cat(sprintf("Simulated  = %.4f\n", mean(replicate(1e4, sim_once()))))
cat(sprintf("Analytic   = %.4f\n", p_shared_birthday(40)))

In [ ]:
# R-2 solution
p_both_red_analytic <- choose(3, 2) / choose(5, 2)

set.seed(2)
sim <- replicate(1e5, {
  draw <- sample(c(rep("R", 3), rep("B", 2)), 2, replace = FALSE)
  all(draw == "R")
})
cat(sprintf("Analytic  P(RR) = %.4f\n", p_both_red_analytic))
cat(sprintf("Simulated P(RR) = %.4f\n", mean(sim)))

In [ ]:
# R-3 solution. We want prior p such that
#   P(D- | T+) = (1-spec)(1-p) / (sens*p + (1-spec)(1-p)) = 0.5
f <- function(p) bayes_dneg_given_tpos(p, sens = 0.95, spec = 0.90) - 0.5
root <- uniroot(f, interval = c(1e-4, 0.999))$root

cat(sprintf("Prevalence that makes P(D- | T+) = 0.5: p ≈ %.4f   (%.2f%%)\n",
            root, root * 100))
cat(sprintf("Check: P(D- | T+) at p = %.4f is %.4f\n",
            root, bayes_dneg_given_tpos(root, 0.95, 0.90)))

## What's next

Chapter 2 introduces **random variables** — functions `X : Ω → ℝ` that turn outcomes into numbers. You'll meet the Bernoulli, binomial, geometric, and Poisson distributions in R via `rbinom`, `rgeom`, `rpois`, and their analytic siblings `dbinom`, `pbinom`, `qbinom`. Everything in this chapter's `notes.md` carries over; we just add a layer of *numerical random quantities* on top of the event-based probability we've built here.